Celem projektu było przetestowanie klasyfikacji kampanii na podstawie wskaźników efektywności. W praktyce zastosowana etykieta została oparta na wskaźniku ROAS, dlatego model w dużej mierze odwzorowywał istniejącą regułę biznesową, a nie odkrywał całkowicie nowych zależności.

Dane wykorzystane w repozytorium są syntetyczne i zostały wygenerowane na potrzeby projektu, tak aby odzwierciedlały przykładowe wyniki kampanii marketingowych bez użycia rzeczywistych danych firmowych.

In [39]:
# pip install plotly
# pip install seaborn
# pip install tensorflow
# pip install nbformat

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense
from tensorflow.keras.activations import elu, exponential, hard_sigmoid, linear, relu, sigmoid, softmax, tanh
from tensorflow.keras.losses import binary_crossentropy, categorical_crossentropy, logcosh, MeanSquaredError, poisson, MeanAbsoluteError
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adadelta, Adam, Nadam, RMSprop, SGD, Ftrl
from sklearn.model_selection import train_test_split
from sklearn import datasets
from keras.datasets import reuters
from sklearn.linear_model import Perceptron
from tensorflow.keras.preprocessing.text import Tokenizer #dot. analizy tekstu
import keras
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
data = pd.read_excel("wyniki_kampanii_syntetyczne.xlsx",header=2)

In [3]:
data_clean = data.dropna()
data_clean


,Kampania,Wyświetlenia,Kliknięcia,CTR,Śr. CPC,Koszt,Konwersje,Wartość konw.,ROAS
0,Campaign_1,4231,114,0.027146,6.14,699.96,4,888.19,1.27
1,Campaign_2,3300,118,0.035951,4.65,548.70,6,223.18,0.41
2,Campaign_3,9651,315,0.032640,3.00,945.00,12,304.71,0.32
3,Campaign_4,29391,1972,0.067113,6.93,13665.96,102,2708.08,0.20
4,Campaign_5,48774,2020,0.041417,7.12,14382.40,41,2394.86,0.17
...,...,...,...,...,...,...,...,...,...
115,Campaign_116,36925,1619,0.043873,3.28,5310.32,160,21499.00,4.05
116,Campaign_117,43260,2789,0.064471,6.60,18407.40,262,67978.50,3.69
117,Campaign_118,43546,2965,0.068102,3.04,9013.60,85,9613.51,1.07
118,Campaign_119,39199,535,0.013649,6.38,3413.30,50,9043.75,2.65


In [5]:
data_clean = data_clean[data_clean['Koszt'] > 0].copy()

data_clean['ROAS'] = data_clean['Wartość konw.'] / data_clean['Koszt']
data_clean['Decyzja'] = ((data_clean['ROAS'] >= 1.5)&(data_clean["Konwersje"]>=3)).astype(int)

X = data_clean[['Kliknięcia', 'Wyświetlenia', 'Śr. CPC', 'CTR', 'Konwersje']]
y = data_clean['Decyzja']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [7]:
X

,Kliknięcia,Wyświetlenia,Śr. CPC,CTR,Konwersje
0,114,4231,6.14,0.027146,4
1,118,3300,4.65,0.035951,6
2,315,9651,3.00,0.032640,12
3,1972,29391,6.93,0.067113,102
4,2020,48774,7.12,0.041417,41
...,...,...,...,...,...
115,1619,36925,3.28,0.043873,160
116,2789,43260,6.60,0.064471,262
117,2965,43546,3.04,0.068102,85
118,535,39199,6.38,0.013649,50


In [8]:
from sklearn.preprocessing import StandardScaler

# skalowanie
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# parametry
learning_rate = 0.01
n_iters = 1000

# funkcja aktywacji
def unit_step_func(x):
    return np.where(x >= 0, 1, 0)

# uczenie
def fit(X, y):
    n_samples, n_features = X.shape
    y = np.array([1 if i > 0 else 0 for i in y])

    weights = np.zeros(n_features)
    bias = 0

    for i in range(n_iters):
        for idx, x_i in enumerate(X):
            linear_output = np.dot(x_i, weights) + bias
            y_predicted = unit_step_func(linear_output)

            update = learning_rate * (y[idx] - y_predicted)
            weights += update * x_i
            bias += update

    return weights, bias

# predykcja
def predict(X, weights, bias):
    linear_output = np.dot(X, weights) + bias
    y_predicted = unit_step_func(linear_output)
    return y_predicted

# accuracy
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

# trening
w, b = fit(X_train, y_train)

# test
y_pred = predict(X_test, w, b)

print("Accuracy:", accuracy(y_test.values, y_pred))


Accuracy: 0.75


In [9]:
print (y.value_counts())

Decyzja
1    69
0    51
Name: count, dtype: int64


In [11]:
data_clean[data_clean['Decyzja']==0][["Kampania","Kliknięcia", "Wyświetlenia","Koszt", "Konwersje","Wartość konw.","Decyzja","ROAS"]]

,Kampania,Kliknięcia,Wyświetlenia,Koszt,Konwersje,Wartość konw.,Decyzja,ROAS
0,Campaign_1,114,4231,699.96,4,888.19,0,1.268915
1,Campaign_2,118,3300,548.70,6,223.18,0,0.406743
2,Campaign_3,315,9651,945.00,12,304.71,0,0.322444
3,Campaign_4,1972,29391,13665.96,102,2708.08,0,0.198162
4,Campaign_5,2020,48774,14382.40,41,2394.86,0,0.166513
7,Campaign_8,2298,40705,10731.66,266,7434.38,0,0.692752
12,Campaign_13,1343,26205,4055.86,41,1874.40,0,0.462146
16,Campaign_17,1478,35952,8882.78,22,1246.54,0,0.140332
18,Campaign_19,1661,42367,2292.18,28,3036.30,0,1.324634
21,Campaign_22,273,31766,1266.72,15,458.87,0,0.362251
